# Bonus B4: How to build a crew with CrewAI

**Notebook 1 of 2: meet CrewAI in a school newsroom**

## Goal

Build a two-person newsroom for a school paper with CrewAI: a sports reporter who can look up Friday's result, and a
headline writer who cannot. Asked alone, the headline writer gets the result wrong. Given the reporter's report
through a task's `context`, it writes the real score, and a check in our own code sends back a headline the paper's
readers would not understand. Then we keep the same crew as files, the way `crewai create crew` lays out a project,
and load it again. Nothing here touches the music shop yet; notebook 2 takes the same objects to the Chinook support
desk.

## What you will learn

- What CrewAI's four objects do: `LLM`, `Agent`, `Task` and `Crew`, and what an agent's role, goal and backstory
  become in the request the model receives.
- How to switch on the crew's own log and read it.
- How a task passes its output to a later task through `context`, so an agent with no tool writes from another
  agent's facts.
- How a guardrail function in our code refuses a task's output and sends the reason back to the agent.
- Why one task is not one model request, and where a crew's request count is kept.
- How a crew is kept as files (`crew.jsonc`, one file per agent, a `tools/` folder), loaded with `load_crew` and run
  with `crewai run`.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name from `models.env`, and
takes the key from `.env` without printing it (lab 00 explains it line by line).

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


CrewAI has its own environment in this lab (its `openai` and `mcp` versions differ from the core labs'), so select
this folder's `.venv` as the kernel. The next cell sets five switches before CrewAI is imported, because CrewAI reads
them when it loads. Each line says what it turns off or where it points.

In [2]:
# CrewAI reads these when it is imported, so they come first
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"      # no usage statistics sent to CrewAI
os.environ["CREWAI_TRACING_ENABLED"] = "false"       # no traces sent to CrewAI's hosted platform
os.environ["CREWAI_DISABLE_VERSION_CHECK"] = "true"  # no check for a newer CrewAI on every start
os.environ["NO_COLOR"] = "1"                         # the crew's log prints without colours

# CrewAI's own files (the last run's task outputs) go to this lab's outputs/ folder, which git ignores
(Path.cwd() / "outputs").mkdir(exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = str(Path.cwd() / "outputs")

---

## Part A: Ask the headline writer with no facts

The school paper, *The Northside Post*, needs a headline for Friday's football match: the Northside Falcons against
the Riverside Otters. The result sheet says the Falcons **lost**, 2 to 3; Lina Haddad scored both Falcons goals, and
the Otters won it in minute 88. We made this result sheet up for the lab, so it is in no model's training data, and our
request does not carry it either.

We first ask for a headline with one plain model call through CrewAI's `LLM` object, the piece every CrewAI agent
uses to talk to a model. There is no agent yet: this plain call stands in for the headline writer. Part C builds the
writer as an agent with no tool, and its `context=[]` run leaves it without facts in the same way.

The `LLM` object takes four settings here:

- `model=f"openai/{MODEL}"`: the `openai/` prefix picks CrewAI's own OpenAI client; the name after it comes from
  `models.env`.
- `api="responses"`: send the requests to OpenAI's Responses API.
- `reasoning_effort=EFFORT`: the effort from `models.env`. CrewAI sends it as `reasoning: {"effort": ...}` on each
  request, and sends nothing when `EFFORT` is empty.
- `timeout=60`: a request that gets stuck fails after 60 seconds instead of freezing the cell.

`llm.call(text)` sends one message and returns the reply as a string.

In [3]:
from crewai import LLM

llm = LLM(model=f"openai/{MODEL}", api="responses", reasoning_effort=EFFORT, timeout=60)

print(llm.call("Write the headline for the school paper about Friday's football match "
               "between the Northside Falcons and the Riverside Otters."))

Falcons Soar Past Otters in Friday Night Football Showdown


In our run: *Falcons Soar Past Otters in Friday Night Football Showdown*. A win for the Falcons, and no score.

**Notice:** nothing in the request said who won. The model wrote a headline anyway, because a headline is what we
asked for, and it filled the gap with a win for the first team in the question. The Falcons lost. The paper cannot
print this, and the facts it needs are on a result sheet that only our code can read.

**Checkpoint A.** One headline, and it does not carry the real result (a Falcons loss, 2 to 3).

---

## Part B: Run one agent and one task as a crew

CrewAI splits the work between agents with different jobs, the way a newsroom splits it between people. Three objects describe that split:

| Object | What it holds | In the newsroom |
|---|---|---|
| `Agent` | a role, a goal, a backstory, an `LLM` and the tools it may use | the sports reporter |
| `Task` | what to do, what the result should look like, and which agent does it | "report Friday's match" |
| `Crew` | the agents and the tasks, run in order by `kickoff` | the newsroom on Friday night |

The reporter needs one tool: a way to read the result sheet.

### A tool the reporter can call

`match_result` is ordinary Python: a dictionary lookup that prints a `TOOL CALLED` line every time it runs, so we can
see when a tool really ran. The `@tool("match_result")` decorator turns the function into a CrewAI tool: it keeps
the name, uses the docstring as the description the model reads, and builds the argument schema from the type hint.
A team that is not on the sheet gets a sentence that says so, not an empty result, so the model has nothing to fill
in.

In [4]:
import json
from crewai.tools import tool

RESULT_SHEET = {
    "Northside Falcons": {
        "played": "Friday",
        "opponent": "Riverside Otters",
        "score": "Falcons 2, Otters 3",
        "falcons_goals": "Lina Haddad (2)",
        "winning_goal": "Otters, in minute 88",
    }
}


@tool("match_result")
def match_result(team: str) -> str:
    """Look up the result of a school team's latest match on the result sheet."""
    print(f"⚽ TOOL CALLED: match_result({team})")
    if team not in RESULT_SHEET:
        return f"no match on the result sheet for {team}"
    return json.dumps(RESULT_SHEET[team])

Before any model is involved, we run the tool ourselves with `.run(...)`, and print what the model will be shown of
it: the name, the description and the argument schema.

In [5]:
print(match_result.run(team="Northside Falcons"))
print(match_result.run(team="Westside Wolves"))
print()
print("name:       ", match_result.name)
print("description:", match_result.description)
print("arguments:  ", match_result.args_schema.model_json_schema())

⚽ TOOL CALLED: match_result(Northside Falcons)
{"played": "Friday", "opponent": "Riverside Otters", "score": "Falcons 2, Otters 3", "falcons_goals": "Lina Haddad (2)", "winning_goal": "Otters, in minute 88"}
⚽ TOOL CALLED: match_result(Westside Wolves)
no match on the result sheet for Westside Wolves

name:        match_result
description: Look up the result of a school team's latest match on the result sheet.
arguments:   {'properties': {'team': {'title': 'Team', 'type': 'string'}}, 'required': ['team'], 'title': 'Match_Result', 'type': 'object'}


The first call printed the `TOOL CALLED` line and returned the Falcons' result as a JSON string; the second, for a
team that is not on the sheet, returned a sentence that says so. The schema has one required string, `team`, built
from the type hint `team: str`. The name, the description and this schema are all the model is shown of our
function: it never sees the Python.

### The reporter, its task and the crew

Every agent in this notebook gets its own `LLM` object, made by `new_llm()` with the same settings as Part A. The
reason is counting: an `LLM` object adds up the requests made through it, and two agents that share one object both
report the shared total, so the crew's count comes out too high.

The task's `description` has `{team}` in it. `kickoff` fills it from `inputs`, so the same task can report any team.
`expected_output` tells the agent what a finished result looks like.

In [6]:
from crewai import Agent, Task, Crew


def new_llm():
    return LLM(model=f"openai/{MODEL}", api="responses", reasoning_effort=EFFORT, timeout=60)


reporter = Agent(
    role="Sports reporter",
    goal="Report school matches exactly as the result sheet records them.",
    backstory="You cover sport for The Northside Post, the paper of Northside School.",
    tools=[match_result],
    llm=new_llm(),
)

report_task = Task(
    description="Report the latest match of the {team}: the opponent, the score and who scored.",
    expected_output="Two or three plain sentences.",
    agent=reporter,
)

print("agent:", reporter.role, "| tools:", [t.name for t in reporter.tools])
print("task: ", report_task.description)
print("      done by", report_task.agent.role, "| expected:", report_task.expected_output)

agent: Sports reporter | tools: ['match_result']
task:  Report the latest match of the {team}: the opponent, the score and who scored.
      done by Sports reporter | expected: Two or three plain sentences.


Nothing has been sent yet: an `Agent` and a `Task` are descriptions. The `Crew` runs them.

`verbose=True` switches on the crew's own log: a box for each step (the crew starting, the task starting, the agent
starting, each tool call, the agent's final answer, the task and the crew finishing). Each box carries an `ID`, a
random identifier CrewAI gives each run and task.

We start the crew with `await crew.akickoff(...)`. Jupyter already runs an event loop, and `crew.kickoff()` inside a
notebook stops with an error that names `kickoff_async()` and `akickoff()`; in a plain Python script, `kickoff()` is
the usual call.

In [7]:
crew = Crew(agents=[reporter], tasks=[report_task], verbose=True)

result = await crew.akickoff(inputs={"team": "Northside Falcons"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3c154bc5-df27-401a-a4fd-ab19902be92e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Report the latest match of the Northside Falcons: the opponent, the score and who scored.                │
│  ID: b9c311c3-55b6-4591-ac7b-c6499809464a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sports reporter                                                                                         │
│                                                                                                                 │
│  Task: Report the latest match of the Northside Falcons: the opponent, the score and who scored.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚽ TOOL CALLED: match_result(Northside Falcons)


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: match_result                                                                                             │
│  Output: {"played": "Friday", "opponent": "Riverside Otters", "score": "Falcons 2, Otters 3", "falcons_goals":  │
│  "Lina Haddad (2)", "winning_goal": "Otters, in minute 88"}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: match_result                                                                                             │
│  Args: {'team': 'Northside Falcons'}                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sports reporter                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The Northside Falcons lost 2–3 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Report the latest match of the Northside Falcons: the opponent, the score and who scored.                │
│  Agent: Sports reporter                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3c154bc5-df27-401a-a4fd-ab19902be92e                                                                       │
│  Final Output: The Northside Falcons lost 2–3 to the Riverside Otters. Lina Haddad scored both goals for the    │
│  Falcons.                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Read the log from the top. The crew starts, the task starts with `{team}` already filled in, and the agent starts.
Then comes our `TOOL CALLED` line, a `Tool Execution Started` box with the arguments the model wrote (`{'team':
'Northside Falcons'}`), and a `Tool Execution Completed` box with what the tool returned. In our run the Completed box printed above the
Started box. CrewAI prints its boxes from events it handles in background threads, so two boxes that come close
together can print out of order. The `#1` in both box titles shows that they belong to the same tool call. The agent's
final answer is the report: the Falcons lost 2–3, and Lina Haddad scored both of their goals. The `ID` lines are new on every run.

### What the run returned

`akickoff` returns a `CrewOutput`. Its `raw` is the last task's text, `tasks_output` holds one `TaskOutput` per task,
and `token_usage` adds up the model requests of the whole run.

In [8]:
print("raw:", result.raw)
print("model requests:", result.token_usage.successful_requests)

raw: The Northside Falcons lost 2–3 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.
model requests: 2


`raw` is the report. **2 model requests for one task**: the first reply asked for `match_result`, CrewAI ran it and
sent back the result, and the second reply wrote the report. A task is one object in our code; how many requests it
takes depends on how many rounds of tool calls its agent needs.

### What the agent's role, goal and backstory became

A `TaskOutput` keeps the messages of its agent's conversation in `messages`. We print each one's role and the start of
its text.

In [9]:
for message in result.tasks_output[0].messages:
    text = message.get("content") or str(message.get("tool_calls"))
    print(f"--- {message['role']}")
    print(text[:400])

--- system
You are Sports reporter. You cover sport for The Northside Post, the paper of Northside School.
Your personal goal is: Report school matches exactly as the result sheet records them.
--- user

Current Task: Report the latest match of the Northside Falcons: the opponent, the score and who scored.

This is the expected criteria for your final answer: Two or three plain sentences.
you MUST return the actual complete content as the final answer, not a summary.
--- assistant
[{'id': 'call_cj5ziK0qmmxVGGcQNdjA2fQI', 'type': 'function', 'function': {'name': 'match_result', 'arguments': '{"team":"Northside Falcons"}'}}]
--- tool
{"played": "Friday", "opponent": "Riverside Otters", "score": "Falcons 2, Otters 3", "falcons_goals": "Lina Haddad (2)", "winning_goal": "Otters, in minute 88"}
--- assistant
The Northside Falcons lost 2–3 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.


Five messages, and the first two are built by CrewAI from our objects:

1. **system**: the agent. *You are Sports reporter.* (the role), then the backstory, then *Your personal goal is:*
   and the goal. The role is a sentence in the prompt; what the agent can do comes from its `tools` list.
2. **user**: the task. The description with `{team}` filled in, then the expected output, then a line CrewAI adds
   to every task.
3. **assistant**: the model's tool call, with the arguments it wrote.
4. **tool**: what `match_result` returned.
5. **assistant**: the report.

**Checkpoint B.** `match_result` returns the Falcons' result for the Falcons and a "no match" sentence for another
team. The log shows the task, a `match_result` call and the agent's final answer, with one `TOOL CALLED` line. The
report carries the real score. The model requests are 2, and the messages run system, user, assistant (the tool call),
tool, assistant.

---

## Part C: Pass the report to the headline writer with `context`

The headline writer from Part A had no facts. It still gets no tool: in a newsroom the headline writer reads the
reporter's copy, not the result sheet. CrewAI's word for that is **context**: a task can list earlier tasks, and their
outputs are added to its request.

![One run of a two-task crew. Our code starts the crew. In task 1 the model asks for the tool, our tool function runs and returns the result, and the model writes it up. Task 1's output is passed on as context to task 2, where the model writes from the context with no tools. Our guardrail function then accepts the output, which becomes the CrewOutput, or refuses it and sends the reason back to task 2's model.](resources/b4_crew_tasks_and_context.png)

Dark boxes are the model's work, light green is our own code, white is text passed along. The guardrail diamond is
Part D.

We build both agents fresh, in two small functions we reuse in Part D. A fresh agent matters here: an agent that has
run in a crew with the log on keeps printing its log boxes in later crews, and these crews run without the log.

In [10]:
def new_reporter():
    return Agent(
        role="Sports reporter",
        goal="Report school matches exactly as the result sheet records them.",
        backstory="You cover sport for The Northside Post, the paper of Northside School.",
        tools=[match_result],
        llm=new_llm(),
    )


def new_writer():
    return Agent(
        role="Headline writer",
        goal="Write short, lively headlines for the school paper.",
        backstory="You write the headlines of The Northside Post, the paper of Northside School. "
                  "You love the words football fans use.",
        llm=new_llm(),
    )


REPORT = "Report the latest match of the {team}: the opponent, the score and who scored."
HEADLINE = "Write the headline for the school paper's story about the latest match of the {team}."

`context=[report_task]` is the new line: the headline task is given the report task's output. Before you run it:
how many model requests will two tasks take?

In [11]:
reporter, writer = new_reporter(), new_writer()

report_task = Task(description=REPORT, expected_output="Two or three plain sentences.", agent=reporter)
headline_task = Task(description=HEADLINE, expected_output="One headline, on one line.", agent=writer,
                     context=[report_task])

crew = Crew(agents=[reporter, writer], tasks=[report_task, headline_task])
result = await crew.akickoff(inputs={"team": "Northside Falcons"})

for output in result.tasks_output:
    print(f"{output.agent}: {output.raw}")
print()
print("context given to the headline task:", headline_task.prompt_context)
print("model requests:", result.token_usage.successful_requests)

⚽ TOOL CALLED: match_result(Northside Falcons)


Sports reporter: The Northside Falcons lost 3–2 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.
Headline writer: Haddad’s Brace Can’t Keep Falcons from Falling 3–2 to Otters

context given to the headline task: The Northside Falcons lost 3–2 to the Riverside Otters. Lina Haddad scored both goals for the Falcons.
model requests: 3


The report says the Falcons lost 3–2, and the headline is *Haddad's Brace Can't Keep Falcons from Falling 3–2 to
Otters*: the real score and the real scorer, written by an agent with no tool. `prompt_context` is the text CrewAI
added to the headline task's request, and it is the report, word for word. **3 model requests for two tasks**: two
for the reporter (the tool call, then the report) and one for the writer, which had nothing to look up.

What fixed the headline is the looked-up result, not the second agent: one agent with `match_result` would also get
the score, in 2 requests, as Part B's reporter got its report. The split buys separate jobs and separate permissions:
the writer never touches the result sheet, and at the desk (notebook 2) the adviser never touches the database. It
costs a request, and the facts are sent twice. Lab 18 measures when a second agent is worth that.

### What happens without `context`

In a sequential crew CrewAI gives a task with no `context` argument the outputs of every task before it, so the line
we wrote is the explicit form of what happens anyway. This notebook does not run a crew without
`context`. To see the default, drop `context=` from the headline task, run the crew again and print
`headline_task.prompt_context`: it holds the report. The default is easy to miss when a crew grows. `context=[]` gives a task no earlier outputs at all, and we can see what
the writer does then.

In [12]:
reporter, writer = new_reporter(), new_writer()

report_task = Task(description=REPORT, expected_output="Two or three plain sentences.", agent=reporter)
headline_task = Task(description=HEADLINE, expected_output="One headline, on one line.", agent=writer,
                     context=[])

result = await Crew(agents=[reporter, writer], tasks=[report_task, headline_task]).akickoff(
    inputs={"team": "Northside Falcons"})

print("report:  ", result.tasks_output[0].raw)
print("headline:", result.raw)
print("context given to the headline task:", repr(headline_task.prompt_context))

⚽ TOOL CALLED: match_result(Northside Falcons)


report:   The Northside Falcons’ latest match was against the Riverside Otters, and the Falcons lost 2–3. Lina Haddad scored both goals for the Falcons.
headline: Falcons Soar to a Big Win!
context given to the headline task: ''


The reporter looked the result up and wrote that the Falcons lost 2–3. The writer was given none of it
(`prompt_context` is empty) and wrote *Falcons Soar to a Big Win!*: a win again, as in Part A. An agent reads another
agent's work only through its task's context.

**Checkpoint C.** With `context=[report_task]`, the headline carries the real score and `prompt_context` holds the
report; two tasks took 3 model requests. With `context=[]`, `prompt_context` is empty and the headline no longer
matches the report, though the reporter looked the result up in the same run.

---

## Part D: Refuse a headline in code with a guardrail

The editor of *The Northside Post* has two rules for headlines: every sports headline carries the score, and no
football slang the younger readers will not know. "Brace", for two goals by one player, is the word the editor sees
most often. These are rules our code can check exactly, so a model is the wrong place to check them.

A **guardrail** is a function CrewAI calls with a task's output before the task counts as done. It returns a pair:

- `(True, value)`: the output is accepted, and `value` becomes the task's result;
- `(False, reason)`: the output is refused, and `reason` is sent back to the same agent, which tries again (three
  times at most, by default).

`check_headline` reads the headline from `output.raw`, prints what it decided, and returns the pair.
`re.findall(r"\d+", headline)` lists the numbers in the headline, so the score check asks for both a 2 and a 3.

In [13]:
import re


def check_headline(output):
    headline = output.raw
    if "brace" in headline.lower():
        print("❌ refused:", headline)
        return (False, "Our readers do not know the word 'brace'. Say 'two goals'.")
    if not {"2", "3"} <= set(re.findall(r"\d+", headline)):
        print("❌ refused:", headline)
        return (False, "A sports headline carries the score.")
    print("✅ accepted:", headline)
    return (True, headline)

A guardrail is plain Python, so we test it on a known output before we hand it to a task: Part C's first headline,
still in `crew.tasks[1].output`.

In [14]:
print(check_headline(crew.tasks[1].output))

❌ refused: Haddad’s Brace Can’t Keep Falcons from Falling 3–2 to Otters
(False, "Our readers do not know the word 'brace'. Say 'two goals'.")


Part C's headline says *Brace*, so the function refused it and returned the reason that CrewAI would send back to the
writer.

Now the same function goes on the headline task with `guardrail=check_headline`. Every attempt the writer makes is
printed by the function, so we see each decision as it happens.

In [15]:
reporter, writer = new_reporter(), new_writer()

report_task = Task(description=REPORT, expected_output="Two or three plain sentences.", agent=reporter)
headline_task = Task(description=HEADLINE, expected_output="One headline, on one line.", agent=writer,
                     context=[report_task], guardrail=check_headline)

result = await Crew(agents=[reporter, writer], tasks=[report_task, headline_task]).akickoff(
    inputs={"team": "Northside Falcons"})

print()
print("headline:", result.raw)
print("model requests:", result.token_usage.successful_requests)

⚽ TOOL CALLED: match_result(Northside Falcons)


❌ refused: Haddad’s Brace Not Enough as Falcons Fall 3–2 to Otters


✅ accepted: Haddad’s Two Goals Not Enough as Falcons Fall 3–2 to Otters

headline: Haddad’s Two Goals Not Enough as Falcons Fall 3–2 to Otters
model requests: 4


Two attempts in one run. The first, *Haddad's Brace Not Enough as Falcons Fall 3–2 to Otters*, was refused; CrewAI
sent the reason back to the writer, and the second, *Haddad's Two Goals Not Enough as Falcons Fall 3–2 to Otters*, was
accepted. The retry cost **one more model request**: 4, against 3 in Part C. The writer's backstory says it loves the
words football fans use, which is why *brace* comes up first; the editor's rule holds because our code checks every
attempt, whatever the prompt says. On a run where the first headline passes, the cell prints one ✅ line and takes 3 requests.

What a ✅ establishes: the headline has a 2 and a 3 in it and does not say *brace*. It does not check who won:
*Falcons Beat Otters 3–2* would pass. A guardrail checks exactly the rules we wrote into it.

**Checkpoint D.** `check_headline` refuses Part C's headline or accepts it, with the reason. On the task, each attempt
prints a ✅ or ❌ line; a refused headline is followed by a second attempt that passes, and the run then takes one model
request more than Part C's.

---

## Part E: Keep the crew as files and run it with `crewai run`

So far the newsroom lives in notebook cells. A CrewAI **project** keeps the same crew in files: one for the crew and
its tasks, one per agent, and a folder of tools. Anyone can then run it from a terminal, and change a role or a task
without touching Python.

`crewai create crew NAME` writes such a project. In a terminal it asks for a model provider, a model and a key (press
Enter to skip the key: a skipped key is not saved), offers to run the new crew, and also starts a git repository and
writes notes for coding assistants. We made `school_paper/` with it and then filled it in for the newsroom, keeping
only the files the crew needs:

| Path | What it holds |
|---|---|
| `school_paper/crew.jsonc` | the crew: its agents by name, the two tasks in order, the default `inputs` |
| `school_paper/agents/reporter.jsonc`, `headline_writer.jsonc` | one agent each: role, goal, backstory, tools |
| `school_paper/tools/match_result.py` | the reporter's tool |
| `school_paper/pyproject.toml` | what `crewai run` installs for the project: `crewai[tools]` 1.15.40, the version the notebooks use. Its `project_id` is a random id that `crewai create` writes; CrewAI adds it to its telemetry and tracing, and the first `export` line below switches both off |

A `.jsonc` file is JSON that may hold `//` comments.

In [16]:
print(Path("school_paper/crew.jsonc").read_text())

{
  // The crew's name, shown in the log
  "name": "school_paper",

  // Each name here has a file in agents/
  "agents": ["reporter", "headline_writer"],

  // The tasks run in this order
  "tasks": [
    {
      "name": "report_task",
      "description": "Report the latest match of the {team}: the opponent, the score and who scored.",
      "expected_output": "Two or three plain sentences.",
      "agent": "reporter"
    },
    {
      "name": "headline_task",
      "description": "Write the headline for the school paper's story about the latest match of the {team}.",
      "expected_output": "One headline, on one line.",
      "agent": "headline_writer",
      // The headline writer is given the report
      "context": ["report_task"]
    }
  ],

  "process": "sequential",
  "verbose": true,

  // Used when no other team is passed in
  "inputs": {
    "team": "Northside Falcons"
  }
}



The two tasks are Part C's, with names (`report_task`, `headline_task`). `"context": ["report_task"]` is Part C's
`context=[report_task]`, written by name. `inputs` gives `{team}` a default value, so `crewai run` needs no
arguments.

In [17]:
for name in ["reporter", "headline_writer"]:
    print(f"--- agents/{name}.jsonc")
    print(Path(f"school_paper/agents/{name}.jsonc").read_text())

--- agents/reporter.jsonc
{
  "role": "Sports reporter",
  "goal": "Report school matches exactly as the result sheet records them.",
  "backstory": "You cover sport for The Northside Post, the paper of Northside School.",

  // "custom:match_result" loads tools/match_result.py
  "tools": ["custom:match_result"]
}

--- agents/headline_writer.jsonc
{
  "role": "Headline writer",
  "goal": "Write short, lively headlines for the school paper.",
  "backstory": "You write the headlines of The Northside Post, the paper of Northside School. You love the words football fans use.",
  "tools": []
}



The agent files hold no model. An agent file without an `llm` field runs on the model named by the `MODEL` environment
variable, or else on CrewAI's default model; an `"llm"` line in the file picks one.

`"custom:match_result"` loads `tools/match_result.py`. It holds the same lookup as Part B, written as a class: a
`BaseTool` subclass with a `name`, a `description` and a `_run` method, the form a project's `tools/` folder uses.

⚠️ Loading a project runs the Python in its `tools/` folder. Run crew projects only from sources you trust.

In [18]:
print(Path("school_paper/tools/match_result.py").read_text())

"""The reporter's tool, in the form a JSON crew project loads: a BaseTool subclass in tools/."""
import json

from crewai.tools import BaseTool

RESULT_SHEET = {
    "Northside Falcons": {
        "played": "Friday",
        "opponent": "Riverside Otters",
        "score": "Falcons 2, Otters 3",
        "falcons_goals": "Lina Haddad (2)",
        "winning_goal": "Otters, in minute 88",
    }
}


class MatchResult(BaseTool):
    name: str = "match_result"
    description: str = "Look up the result of a school team's latest match on the result sheet."

    def _run(self, team: str) -> str:
        print(f"TOOL CALLED: match_result({team})")
        if team not in RESULT_SHEET:
            return f"no match on the result sheet for {team}"
        return json.dumps(RESULT_SHEET[team])



### Load the project in Python

`load_crew` reads `crew.jsonc` and the agent files and returns two things: a `Crew`, and the default `inputs` from
the file. We print the objects it built, the same kinds as Parts B to D.

In [19]:
from crewai.project import load_crew

paper_crew, defaults = load_crew("school_paper/crew.jsonc")

print(type(paper_crew).__name__, "| default inputs:", defaults)
for agent in paper_crew.agents:
    print("agent:", agent.role, "| tools:", [t.name for t in agent.tools])
for task in paper_crew.tasks:
    context = [t.name for t in task.context] if isinstance(task.context, list) else "not set"
    print("task: ", task.name, "| done by", task.agent.role, "| context:", context)

Crew | default inputs: {'team': 'Northside Falcons'}
agent: Sports reporter | tools: ['match_result']
agent: Headline writer | tools: []
task:  report_task | done by Sports reporter | context: not set
task:  headline_task | done by Headline writer | context: ['report_task']


`load_crew` built the same objects as Part C: two agents, `match_result` on the reporter only, and two tasks, the
second with the first as its context. The report task's context is not set, which in a sequential crew means the
outputs of the tasks before it, and there are none.

We give each loaded agent its own `new_llm()`, so the run uses the effort and the time limit of this notebook, and
start it with the default inputs. `crew.jsonc` says `"verbose": true`, so the crew's log prints, the same log
`crewai run` prints in a terminal.

In [20]:
for agent in paper_crew.agents:
    agent.llm = new_llm()

result = await paper_crew.akickoff(inputs=defaults)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: school_paper                                                                                             │
│  ID: 55e63ff2-aa5b-4677-8393-d16e3e682dfe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: report_task                                                                                              │
│  ID: 4636c6eb-a63b-454c-9159-94d6c0236037                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Sports reporter                                                                                         │
│                                                                                                                 │
│  Task: Report the latest match of the Northside Falcons: the opponent, the score and who scored.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

TOOL CALLED: match_result(Northside Falcons)


╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: match_result                                                                                             │
│  Args: {'team': 'Northside Falcons'}                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: match_result                                                                                             │
│  Output: {"played": "Friday", "opponent": "Riverside Otters", "score": "Falcons 2, Otters 3", "falcons_goals":  │
│  "Lina Haddad (2)", "winning_goal": "Otters, in minute 88"}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (3 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Headline writer                                                                                         │
│                                                                                                                 │
│  Task: Write the headline for the school paper's story about the latest match of the Northside Falcons.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Headline writer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Haddad’s Brace Can’t Keep Falcons from Falling 3–2 to Otters                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: headline_task                                                                                            │
│  Agent: Headline writer                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: school_paper                                                                                             │
│  ID: 55e63ff2-aa5b-4677-8393-d16e3e682dfe                                                                       │
│  Final Output: Haddad’s Brace Can’t Keep Falcons from Falling 3–2 to Otters                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [21]:
print("headline:", result.raw)
print("model requests:", result.token_usage.successful_requests)

headline: Haddad’s Brace Can’t Keep Falcons from Falling 3–2 to Otters
model requests: 3


The log is Part B's, now for two tasks, and the run took **3 model requests**, as in Part C. The headline says
*Brace* again: Part D's guardrail is a function in this notebook, not part of the project, so nothing refused the
word. The tool boxes say `#2`: since the kernel started, CrewAI has
numbered each tool's calls only in runs that print the log. Part B's call was `#1`, and Parts C and D ran without the
log.

### Run the project from a terminal

`crewai run` is the project's own way to start. In a terminal, with this lab's environment active (`source
.venv/bin/activate` in `bonus/B4_CrewAI_Crew`):

```bash
cd school_paper
export CREWAI_DISABLE_TELEMETRY=true CREWAI_TRACING_ENABLED=false
export MODEL="openai/$(grep '^MODEL_DEFAULT=' ../../../models.env | cut -d= -f2)"
crewai run
```

The notebook's settings do not reach a terminal, so the two `export` lines set what the run needs there. The first
sets the two switches that stop data going to CrewAI; the second reads the model name from `models.env`. Without
`MODEL`, the agent files fall back to CrewAI's
default model, not the one in `models.env`. Either way the terminal run sends no reasoning effort, so the API's
default effort applies.

What it does, in order:

1. Reads `pyproject.toml` and, the first time, installs the project's own environment in `school_paper/.venv` with
   `uv`. The project asks for `crewai[tools]`, a large download (about 850 MB).
2. Checks that the key each agent's model needs is set: in the environment, in `school_paper/.env`, or in the nearest
   `.env` in a folder above. The repository's `.env` is such a folder above, so nothing more is needed.
3. Runs the crew with its default inputs and prints the log we saw above, ending with the headline.

⚠️ Start `crewai run` from a terminal, not from a notebook cell: its first run installs packages (about 850 MB), and no
cell in this lab installs packages.

**Checkpoint E.** `crew.jsonc` with two tasks and `"context": ["report_task"]`; two agent files and the tool class;
`load_crew` builds a `Crew` with the two agents (`match_result` on the reporter only), the two tasks and the default
inputs `{'team': 'Northside Falcons'}`; the run prints the log and ends with a headline that carries the score.

---

## What you can do now

- Make an `LLM`, call it once, and give each agent its own (Part A, Part B).
- Build an `Agent` with a tool, a `Task` with `{placeholders}`, and run them as a `Crew` with `await
  crew.akickoff(inputs=...)`, with the crew's log on (Part B).
- Read a run from `CrewOutput`: `raw`, `tasks_output`, each task's `messages`, and `token_usage` (Part B).
- Pass one task's output to another with `context`, and say what a sequential crew does when `context` is left out
  (Part C).
- Refuse a task's output with a guardrail function and send the reason back (Part D).
- Keep a crew as a project, load it with `load_crew`, and run it with `crewai run` (Part E).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `RuntimeError: Agent execution was invoked synchronously from within a running event loop.` | `crew.kickoff()` was called in a notebook, which already runs an event loop | write `await crew.akickoff(...)` in a notebook; keep `kickoff()` for scripts |
| `ModuleNotFoundError: No module named 'crewai'` | the kernel is the repository's core environment | select `bonus/B4_CrewAI_Crew/.venv` as the kernel (`uv sync --locked --python 3.13` in this folder first) |
| log boxes from an agent you built for a quiet crew | the agent ran earlier in a crew with `verbose=True` | build the agent again (`new_reporter()`, `new_writer()`) |
| `Exception: Task failed guardrail validation after 3 retries.` | every attempt was refused | read the ❌ lines: if the rule is one the model keeps missing, say it in the task's description too, or raise `guardrail_max_retries` |
| `OPENAI_API_KEY is not set` from `crewai run` | no key in the environment or in a `.env` the CLI found | put the key in the repository's `.env`, or add `OPENAI_API_KEY=...` to `school_paper/.env` (its `.gitignore` keeps it out of git) |

## Challenge

Add a third member to the newsroom: a social media editor who writes one post about the match.

1. Write a `new_editor()` agent with no tools, and a task whose `context` lists **both** earlier tasks, so the editor
   reads the report and the headline.
2. Give the task a guardrail that refuses a post that does not name Lina Haddad or is longer than 280 characters.
3. Run the three tasks as one crew and print each task's output and the model requests.

Expected: three task outputs, a post that names Lina Haddad and carries the score, and more model requests than tasks.
Look at `prompt_context` of the editor's task: what is in it, and in what order?

The worked solution, run with its outputs, is in `solved/b4a_meet_crewai_solution.ipynb`.

## Going further: steps in an order our code sets, with a Flow

A crew lets the agents' tasks run in the order of the list. CrewAI's other object, the **Flow**, runs steps in an
order our code sets: `@start()` marks the first method, and `@listen(method)` runs a method when another one
finishes, with its return value. A step can be plain Python or start a crew. This one uses no model: it reads the
result sheet, then builds a line from it.

`suppress_flow_events=True` keeps the flow's own log boxes out of the output. `flow.plot(name, show=False)` draws
the flow as an HTML page and returns where it saved the page (a temporary folder on our machine; `show=True`, the
default, also opens it in the browser). We print only the file's name; print `page` to see the full path on yours.

In [22]:
from crewai.flow.flow import Flow, start, listen


class Edition(Flow):
    @start()
    def read_result(self):
        return json.loads(match_result.run(team="Northside Falcons"))

    @listen(read_result)
    def write_line(self, sheet):
        return f"Friday: {sheet['score']}. Falcons goals: {sheet['falcons_goals']}."


flow = Edition(suppress_flow_events=True)
print(await flow.akickoff())

page = flow.plot("edition_flow.html", show=False)
print("drawing saved as:", Path(page).name)

⚽ TOOL CALLED: match_result(Northside Falcons)
Friday: Falcons 2, Otters 3. Falcons goals: Lina Haddad (2).
drawing saved as: edition_flow.html


The flow ran its two methods in order: `read_result` called the tool (one `TOOL CALLED` line) and returned the sheet,
and `write_line` built the line from it. No model request was made. Open the saved page (the full path is in
`page`) in a browser to see the two methods and the arrow between them.

## Resources

- [CrewAI documentation](https://docs.crewai.com/): the home page, with a quickstart
- [Agents](https://docs.crewai.com/en/concepts/agents): role, goal, backstory, tools, `llm`
- [Tasks](https://docs.crewai.com/en/concepts/tasks): `description`, `expected_output`, `context`, guardrails
- [Crews](https://docs.crewai.com/en/concepts/crews): `kickoff`, `akickoff`, `verbose`, usage
- [LLMs](https://docs.crewai.com/en/concepts/llms): the `LLM` object and its settings
- [Tools](https://docs.crewai.com/en/concepts/tools): the `@tool` decorator and `BaseTool`
- [CLI](https://docs.crewai.com/en/concepts/cli): `crewai create crew` and `crewai run`
- [Flows](https://docs.crewai.com/en/concepts/flows): `@start`, `@listen`, `plot`
- [Telemetry](https://docs.crewai.com/en/telemetry): what CrewAI collects and how to switch it off

Next: notebook 2, `b4b_crewai_at_the_desk.ipynb`, takes the same objects to the Chinook support desk.